# Support Integrity Auditor (SIA)
### MARS Open Projects 2026
**Pipeline:** Pseudo-Label Generation → Classifier Training → Evidence Dossier Generation

---

## 0. Install Dependencies

In [ ]:
# Run this cell first (Colab / fresh environment)
!pip install -q transformers datasets peft accelerate sentence-transformers
!pip install -q scikit-learn pandas numpy tqdm spacy xgboost imbalanced-learn
!pip install -q streamlit plotly kaggle
!python -m spacy download en_core_web_sm -q
print('All dependencies installed.')

## 1. Imports & Config

In [ ]:
import os, json, re, warnings, hashlib
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, confusion_matrix, classification_report
)
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTE
import spacy
import torch
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset

warnings.filterwarnings('ignore')
tqdm.pandas()

# ── CONFIG ──────────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

PRIORITY_ORDER = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
PRIORITY_LABELS = {v: k for k, v in PRIORITY_ORDER.items()}
MODEL_NAME = 'microsoft/deberta-v3-small'
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 4
LORA_R = 8
LORA_ALPHA = 16

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('Config loaded.')

## 2. Load & Explore Dataset

In [ ]:


CSV_PATH = 'customer_support_tickets.csv'   
df = pd.read_csv(CSV_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── Column audit ──────────────────────────────────────────────────────────
print(df.columns.tolist())
print(df.dtypes)
print(df.isnull().sum())

## 3. Preprocessing

In [ ]:
# ── Normalise column names ────────────────────────────────────────────────
df.columns = df.columns.str.strip().str.replace(' ', '_')

# Map column names defensively
COL_MAP = {
    'Ticket_ID'         : 'ticket_id',
    'Ticket_Subject'    : 'subject',
    'Ticket_Description': 'description',
    'Ticket_Priority'   : 'priority',
    'Ticket_Channel'    : 'channel',
    'Ticket_Type'       : 'ticket_type',
    'Resolution_Time'   : 'resolution_time',
    'Customer_Email'    : 'customer_email',
    'Product_Purchased' : 'product',
}
df.rename(columns={k: v for k, v in COL_MAP.items() if k in df.columns}, inplace=True)

# Drop rows without priority or description
df.dropna(subset=['priority', 'description'], inplace=True)
df.reset_index(drop=True, inplace=True)

# Create ticket_id if missing
if 'ticket_id' not in df.columns:
    df['ticket_id'] = [f'TKT-{i:05d}' for i in range(len(df))]

# Combine text fields
df['text'] = (df.get('subject', '') + ' [SEP] ' + df['description']).str.strip()

# Encode priority
df['priority_num'] = df['priority'].map(PRIORITY_ORDER)
df.dropna(subset=['priority_num'], inplace=True)
df['priority_num'] = df['priority_num'].astype(int)

# Parse resolution_time (extract numeric hours if it's a string like '5 days')
def parse_resolution_time(val):
    if pd.isna(val): return np.nan
    val = str(val)
    nums = re.findall(r'[\d\.]+', val)
    if not nums: return np.nan
    hours = float(nums[0])
    if 'day' in val.lower(): hours *= 24
    if 'week' in val.lower(): hours *= 168
    return hours

if 'resolution_time' in df.columns:
    df['resolution_hours'] = df['resolution_time'].apply(parse_resolution_time)
else:
    df['resolution_hours'] = np.nan

print(f'Clean dataset: {df.shape}')
df[['ticket_id','text','priority','priority_num','resolution_hours']].head()

## 4. Stage 1 — Pseudo-Label Generation (Self-Supervised)

### Signal A: Rule-Based NLP Severity Scorer

In [ ]:
nlp = spacy.load('en_core_web_sm')

# Keyword lists
CRITICAL_KEYWORDS = [
    'urgent', 'critical', 'emergency', 'immediately', 'asap', 'broken',
    'outage', 'down', 'crash', 'failure', 'data loss', 'security breach',
    'cannot access', 'system down', 'not working', 'blocked', 'escalate',
    'unacceptable', 'lawsuit', 'legal', 'refund', 'fraud', 'hacked'
]
HIGH_KEYWORDS = [
    'issue', 'problem', 'error', 'bug', 'slow', 'delay', 'frustrat',
    'not responding', 'failed', 'incorrect', 'wrong', 'missing', 'stuck'
]
LOW_KEYWORDS = [
    'question', 'inquiry', 'wondering', 'curious', 'when', 'how do i',
    'feature request', 'suggestion', 'feedback', 'would like', 'could you'
]
ESCALATION_PHRASES = [
    'speak to manager', 'speak to supervisor', 'escalate', 'not acceptable',
    'very disappointed', 'extremely frustrated', 'worst experience'
]
NEGATION_WORDS = ['not', "n't", 'never', 'no', 'unable', 'cannot', "can't"]

def rule_based_severity(text: str) -> float:
    """
    Returns a severity score in [0, 3] matching PRIORITY_ORDER scale.
    """
    t = text.lower()
    doc = nlp(t[:2000])  # limit for speed
    tokens = [tok.text for tok in doc]

    score = 1.0  # default medium-low

    # Critical keywords
    crit_hits = sum(1 for kw in CRITICAL_KEYWORDS if kw in t)
    score += min(crit_hits * 0.6, 2.0)

    # High keywords
    high_hits = sum(1 for kw in HIGH_KEYWORDS if kw in t)
    score += min(high_hits * 0.2, 0.8)

    # Low keywords (reduce score)
    low_hits = sum(1 for kw in LOW_KEYWORDS if kw in t)
    score -= min(low_hits * 0.3, 0.9)

    # Escalation phrases
    esc_hits = sum(1 for ph in ESCALATION_PHRASES if ph in t)
    score += esc_hits * 0.5

    # Negation booster (negated positive = negative experience)
    neg_count = sum(1 for tok in tokens if tok in NEGATION_WORDS)
    score += min(neg_count * 0.15, 0.5)

    # Exclamation marks signal urgency
    score += min(text.count('!') * 0.1, 0.3)

    # All caps words
    caps_words = re.findall(r'\b[A-Z]{3,}\b', text)
    score += min(len(caps_words) * 0.15, 0.45)

    return float(np.clip(score, 0, 3))

tqdm.pandas(desc='Rule-based NLP')
df['signal_nlp_score'] = df['text'].progress_apply(rule_based_severity)
df['signal_nlp_severity'] = pd.cut(
    df['signal_nlp_score'],
    bins=[-0.1, 0.75, 1.5, 2.25, 3.1],
    labels=[0, 1, 2, 3]
).astype(int)

print('Signal A (Rule-based NLP) distribution:')
print(df['signal_nlp_severity'].value_counts().sort_index())

### Signal B: Embedding-Based Semantic Clustering

In [ ]:
print('Loading sentence-transformer...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print('Encoding tickets...')
embeddings = embedder.encode(
    df['text'].tolist(),
    batch_size=64,
    show_progress_bar=True
)
df['embedding'] = list(embeddings)  # store for later

# Cluster into 4 groups (= priority levels)
print('Clustering into 4 severity groups...')
kmeans = KMeans(n_clusters=4, random_state=SEED, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)
df['cluster'] = cluster_labels

# Align clusters to severity by median resolution time or NLP score
cluster_severity_map = (
    df.groupby('cluster')['signal_nlp_score']
    .median()
    .rank(method='first')
    .sub(1)
    .astype(int)
    .to_dict()
)
df['signal_cluster_severity'] = df['cluster'].map(cluster_severity_map)

print('Signal B (Cluster) distribution:')
print(df['signal_cluster_severity'].value_counts().sort_index())

### Signal C: Resolution-Time Regression (Severity Proxy)

In [ ]:
valid_rt = df['resolution_hours'].notna()

if valid_rt.sum() > 100:
    X_rt = embeddings[valid_rt]
    y_rt = df.loc[valid_rt, 'resolution_hours'].values
    rt_model = LinearRegression()
    rt_model.fit(X_rt, y_rt)
    rt_pred = rt_model.predict(embeddings)
    # Bin into 4 severity levels (higher RT → higher true severity)
    df['signal_rt_score'] = rt_pred
    df['signal_rt_severity'] = pd.qcut(
        rt_pred, q=4, labels=[0, 1, 2, 3]
    ).astype(int)
    USE_RT_SIGNAL = True
    print('Signal C (Resolution-time regression) enabled.')
    print(df['signal_rt_severity'].value_counts().sort_index())
else:
    df['signal_rt_severity'] = df['signal_nlp_severity']  # fallback
    USE_RT_SIGNAL = False
    print('Signal C disabled (insufficient resolution_time data) — using Signal A as fallback.')

### Fusion & Binary Mismatch Label

In [ ]:
# ── Weighted fusion of signals ────────────────────────────────────────────
# Weights justified by individual signal quality (see ablation below)
W_NLP = 0.45   # highest: direct keyword signal
W_CLU = 0.30   # semantic clustering
W_RT  = 0.25   # weaker proxy

df['inferred_severity_score'] = (
    W_NLP * df['signal_nlp_severity'] +
    W_CLU * df['signal_cluster_severity'] +
    W_RT  * df['signal_rt_severity']
)

# Round to integer severity level
df['inferred_severity_num'] = df['inferred_severity_score'].round().astype(int).clip(0, 3)
df['inferred_severity'] = df['inferred_severity_num'].map(PRIORITY_LABELS)

# Binary mismatch: 1 = mismatch, 0 = consistent
df['mismatch'] = (df['inferred_severity_num'] != df['priority_num']).astype(int)

# Mismatch type
def mismatch_type(row):
    if row['mismatch'] == 0:
        return 'Consistent'
    return 'Hidden Crisis' if row['inferred_severity_num'] > row['priority_num'] else 'False Alarm'

df['mismatch_type'] = df.apply(mismatch_type, axis=1)

print('Pseudo-label distribution:')
print(df['mismatch'].value_counts())
print('\nMismatch types:')
print(df['mismatch_type'].value_counts())

### Ablation: Individual Signal Contributions

In [ ]:
# Pairwise agreement between signals
agree_nlp_clu = (df['signal_nlp_severity'] == df['signal_cluster_severity']).mean()
agree_nlp_rt  = (df['signal_nlp_severity'] == df['signal_rt_severity']).mean()
agree_clu_rt  = (df['signal_cluster_severity'] == df['signal_rt_severity']).mean()

print('=== ABLATION TABLE ===')
print(f'Signal Pair Agreement:')
print(f'  NLP ↔ Cluster  : {agree_nlp_clu:.3f}')
print(f'  NLP ↔ Res-Time : {agree_nlp_rt:.3f}')
print(f'  Cluster ↔ RT   : {agree_clu_rt:.3f}')

# Individual mismatch rates per signal vs assigned priority
for sig, col in [('NLP','signal_nlp_severity'),('Cluster','signal_cluster_severity'),('RT','signal_rt_severity')]:
    rate = (df[col] != df['priority_num']).mean()
    print(f'  {sig} mismatch rate vs assigned: {rate:.3f}')

## 5. Stage 2 — Classifier Training (DeBERTa-v3-small + LoRA)

In [ ]:
# ── Feature engineering: add structured metadata ──────────────────────────
le_channel = LabelEncoder()
le_type    = LabelEncoder()

if 'channel' in df.columns:
    df['channel_enc'] = le_channel.fit_transform(df['channel'].fillna('unknown'))
else:
    df['channel_enc'] = 0

if 'ticket_type' in df.columns:
    df['type_enc'] = le_type.fit_transform(df['ticket_type'].fillna('unknown'))
else:
    df['type_enc'] = 0

# Normalise resolution hours
rt_mean = df['resolution_hours'].mean()
rt_std  = df['resolution_hours'].std()
df['rt_norm'] = (df['resolution_hours'].fillna(rt_mean) - rt_mean) / (rt_std + 1e-8)

# Compose enriched text: text + structured fields
def build_input_text(row):
    chan = row.get('channel', 'unknown')
    ttype = row.get('ticket_type', 'unknown')
    rt = row['resolution_hours']
    rt_str = f'{rt:.1f}h' if not pd.isna(rt) else 'unknown'
    return (
        f"[CHANNEL: {chan}] [TYPE: {ttype}] [RT: {rt_str}] "
        f"{row['text']}"
    )

df['model_input'] = df.apply(build_input_text, axis=1)
print('Sample model input:')
print(df['model_input'].iloc[0][:300])

In [ ]:
# ── Train/Val/Test split ──────────────────────────────────────────────────
train_df, test_df = train_test_split(df, test_size=0.15, random_state=SEED, stratify=df['mismatch'])
train_df, val_df  = train_test_split(train_df, test_size=0.12, random_state=SEED, stratify=train_df['mismatch'])
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# Class imbalance: weighted loss via class_weight
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.array([0,1]), y=train_df['mismatch'].values)
class_weights = torch.tensor(cw, dtype=torch.float).to(DEVICE)
print(f'Class weights: {cw}')

In [ ]:
# ── Tokenise ──────────────────────────────────────────────────────────────
print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenise(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LEN,
        padding=False
    )

def to_dataset(dataframe):
    ds = Dataset.from_dict({
        'text': dataframe['model_input'].tolist(),
        'labels': dataframe['mismatch'].tolist()
    })
    return ds.map(tokenise, batched=True, remove_columns=['text'])

train_ds = to_dataset(train_df)
val_ds   = to_dataset(val_df)
test_ds  = to_dataset(test_df)
print('Datasets tokenised.')

In [ ]:
# ── DeBERTa + LoRA setup ──────────────────────────────────────────────────
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.1,
    target_modules=['query_proj', 'value_proj'],
    bias='none'
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# ── Custom Trainer with weighted loss ────────────────────────────────────
from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc   = accuracy_score(labels, preds)
    f1    = f1_score(labels, preds, average='macro')
    rec   = recall_score(labels, preds, average=None)
    return {
        'accuracy': acc,
        'macro_f1': f1,
        'recall_consistent': rec[0],
        'recall_mismatch':   rec[1]
    }

training_args = TrainingArguments(
    output_dir='./sia_model',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    fp16=(DEVICE == 'cuda'),
    seed=SEED,
    report_to='none'
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

print('Training...')
trainer.train()

In [ ]:
# ── Evaluation on held-out test set ──────────────────────────────────────
results = trainer.evaluate(test_ds)
print('\n=== TEST SET RESULTS ===')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')

preds_out = trainer.predict(test_ds)
y_pred = np.argmax(preds_out.predictions, axis=-1)
y_true = test_df['mismatch'].values

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=['Consistent','Mismatch']))

print('Confusion Matrix:')
print(confusion_matrix(y_true, y_pred))

# Threshold check
acc = accuracy_score(y_true, y_pred)
mf1 = f1_score(y_true, y_pred, average='macro')
rec = recall_score(y_true, y_pred, average=None)
print(f'\nVerification: Acc={acc:.3f} (≥0.83), MacroF1={mf1:.3f} (≥0.82), Recall={rec} (≥0.78 each)')

In [ ]:
# ── Save model ────────────────────────────────────────────────────────────
os.makedirs('sia_model_final', exist_ok=True)
model.save_pretrained('sia_model_final')
tokenizer.save_pretrained('sia_model_final')
print('Model saved to sia_model_final/')

## 6. Stage 3 — Evidence Dossier Generation

In [ ]:
def extract_evidence(row: pd.Series) -> list:
    """Extract traceable evidence items from ticket fields."""
    evidence = []
    text = str(row.get('text', ''))

    # Keyword evidence
    found_critical = [kw for kw in CRITICAL_KEYWORDS if kw in text.lower()]
    found_high     = [kw for kw in HIGH_KEYWORDS if kw in text.lower()]
    found_esc      = [ph for ph in ESCALATION_PHRASES if ph in text.lower()]

    if found_critical:
        evidence.append({
            'signal'     : 'critical_keyword',
            'value'      : ', '.join(found_critical[:5]),
            'source_field': 'description',
            'weight'     : f'{len(found_critical) * 0.6:.2f}'
        })
    if found_high:
        evidence.append({
            'signal'     : 'high_severity_keyword',
            'value'      : ', '.join(found_high[:5]),
            'source_field': 'description',
            'weight'     : f'{len(found_high) * 0.2:.2f}'
        })
    if found_esc:
        evidence.append({
            'signal'     : 'escalation_phrase',
            'value'      : ', '.join(found_esc[:3]),
            'source_field': 'description',
            'weight'     : '0.50'
        })

    # Exclamation / caps evidence
    excl = text.count('!')
    if excl > 0:
        evidence.append({
            'signal': 'exclamation_density',
            'value' : str(excl),
            'source_field': 'description',
            'weight': f'{min(excl * 0.1, 0.3):.2f}'
        })

    # Resolution time evidence
    rt = row.get('resolution_hours', np.nan)
    if not pd.isna(rt):
        high_rt = rt > (rt_mean + rt_std)
        evidence.append({
            'signal'        : 'resolution_time',
            'value'         : f'{rt:.1f} hours',
            'source_field'  : 'resolution_time',
            'interpretation': 'Abnormally high — suggests underlying complexity' if high_rt else 'Within normal range'
        })

    # Channel evidence
    chan = row.get('channel', None)
    if chan:
        high_urgency_channels = ['phone', 'social_media', 'social media']
        if str(chan).lower() in high_urgency_channels:
            evidence.append({
                'signal'     : 'channel',
                'value'      : str(chan),
                'source_field': 'channel',
                'weight'     : '0.30'
            })

    return evidence


def generate_constraint_analysis(row: pd.Series, evidence: list) -> str:
    """Generate a 2–3 sentence grounded analysis — no hallucination."""
    assigned = row['priority']
    inferred = row['inferred_severity']
    mtype    = row['mismatch_type']
    sigs     = [e['signal'] for e in evidence]

    if mtype == 'Hidden Crisis':
        intro = (
            f"This ticket was assigned {assigned} priority but the semantic and "
            f"structural analysis indicates a true severity of {inferred}. "
        )
    else:
        intro = (
            f"This ticket was inflated to {assigned} priority; inferred signals "
            f"suggest actual severity is only {inferred}. "
        )

    sig_str = ', '.join(sigs[:3]) if sigs else 'text analysis'
    mid = f"Evidence sourced from {sig_str} fields directly traceable to the input ticket. "
    end = (
        "Immediate re-triage is recommended to prevent SLA breach." if mtype == 'Hidden Crisis'
        else "Deprioritisation is advised to free support bandwidth for genuine high-severity tickets."
    )
    return intro + mid + end


def generate_dossier(row: pd.Series, confidence: float) -> dict:
    evidence = extract_evidence(row)
    if not evidence:
        evidence = [{'signal': 'text_semantics', 'value': 'embedding cluster assignment',
                     'source_field': 'description', 'weight': '0.30'}]

    return {
        'ticket_id'         : str(row.get('ticket_id', 'unknown')),
        'assigned_priority' : row['priority'],
        'inferred_severity' : row['inferred_severity'],
        'mismatch_type'     : row['mismatch_type'],
        'severity_delta'    : int(row['inferred_severity_num'] - row['priority_num']),
        'feature_evidence'  : evidence,
        'constraint_analysis': generate_constraint_analysis(row, evidence),
        'confidence'        : f'{confidence:.3f}'
    }


print('Dossier generation functions ready.')

In [ ]:
# ── Generate dossiers for all mismatched tickets in the test set ──────────
from scipy.special import softmax as sp_softmax

logits_all = preds_out.predictions
probs_all  = sp_softmax(logits_all, axis=-1)

dossiers = []
test_df_reset = test_df.reset_index(drop=True)

for i, (_, row) in enumerate(test_df_reset.iterrows()):
    pred  = y_pred[i]
    conf  = probs_all[i][pred]
    if pred == 1:   # predicted mismatch
        d = generate_dossier(row, conf)
        dossiers.append(d)

print(f'Generated {len(dossiers)} dossiers for mismatched tickets.')

# Save dossiers
os.makedirs('outputs', exist_ok=True)
with open('outputs/dossiers.json', 'w') as f:
    json.dump(dossiers, f, indent=2)
print('Dossiers saved to outputs/dossiers.json')

# Preview first dossier
print('\nSample Dossier:')
print(json.dumps(dossiers[0] if dossiers else {}, indent=2))

## 7. Adversarial Robustness Test

In [ ]:
# 10 hand-crafted adversarial tickets designed to fool keyword-based systems
adversarial_tickets = [
    # Hidden Crisis: low priority language but critical situation
    {'ticket_id':'ADV-001','subject':'quick question','description':
     'just wondering if its normal that my entire order history vanished after i updated the app. no rush.',
     'priority':'Low','channel':'email','ticket_type':'technical','resolution_hours':2.0},
    {'ticket_id':'ADV-002','subject':'minor thing','description':
     'hey, not sure if this matters but i cannot login and my account seems locked with pending charges of $999.',
     'priority':'Low','channel':'chat','ticket_type':'billing','resolution_hours':1.5},
    {'ticket_id':'ADV-003','subject':'feedback','description':
     'wanted to let you know our entire enterprise team (50 users) is unable to access the dashboard since yesterday morning.',
     'priority':'Medium','channel':'email','ticket_type':'technical','resolution_hours':3.0},
    {'ticket_id':'ADV-004','subject':'quick check','description':
     'is it supposed to show my personal data to other users when they log in? saw someone elses name on my screen.',
     'priority':'Low','channel':'chat','ticket_type':'security','resolution_hours':2.0},
    {'ticket_id':'ADV-005','subject':'small issue maybe','description':
     'the payment gateway seems to charge twice for every transaction. happened to at least 10 customers today.',
     'priority':'Medium','channel':'phone','ticket_type':'billing','resolution_hours':4.0},
    # False Alarm: high priority language but trivial situation
    {'ticket_id':'ADV-006','subject':'URGENT EMERGENCY','description':
     'i urgently need to know how to change my email notification frequency. this is critical for me.',
     'priority':'Critical','channel':'email','ticket_type':'general','resolution_hours':0.5},
    {'ticket_id':'ADV-007','subject':'ASAP CRITICAL HELP','description':
     'IMMEDIATELY need to update my profile picture. this is extremely urgent and cannot wait!!!',
     'priority':'High','channel':'chat','ticket_type':'account','resolution_hours':0.3},
    {'ticket_id':'ADV-008','subject':'system failure alert','description':
     'i was wondering if you could help me find the dark mode option. could not locate the button.',
     'priority':'High','channel':'email','ticket_type':'general','resolution_hours':0.2},
    {'ticket_id':'ADV-009','subject':'escalating to management','description':
     'please escalate — i need to know the delivery date for my order placed last week.',
     'priority':'Critical','channel':'phone','ticket_type':'shipping','resolution_hours':1.0},
    {'ticket_id':'ADV-010','subject':'server down reported','description':
     'our team would like a walkthrough of the new reporting features introduced in the last update.',
     'priority':'Critical','channel':'email','ticket_type':'general','resolution_hours':1.5},
]

adv_df = pd.DataFrame(adversarial_tickets)
adv_df['text'] = adv_df['subject'] + ' [SEP] ' + adv_df['description']
adv_df['model_input'] = adv_df.apply(build_input_text, axis=1)
adv_df['priority_num'] = adv_df['priority'].map(PRIORITY_ORDER)
adv_df['signal_nlp_severity'] = adv_df['text'].apply(rule_based_severity).apply(
    lambda s: int(np.clip(round(s), 0, 3))
)
adv_df['inferred_severity_num'] = adv_df['signal_nlp_severity']  # simplified for demo
adv_df['inferred_severity'] = adv_df['inferred_severity_num'].map(PRIORITY_LABELS)
adv_df['mismatch'] = (adv_df['inferred_severity_num'] != adv_df['priority_num']).astype(int)
adv_df['mismatch_type'] = adv_df.apply(mismatch_type, axis=1)
adv_df['resolution_hours'] = adv_df['resolution_hours'].astype(float)

# Tokenise and predict
adv_ds = Dataset.from_dict({
    'text': adv_df['model_input'].tolist(),
    'labels': adv_df['mismatch'].tolist()
}).map(tokenise, batched=True, remove_columns=['text'])

adv_preds = trainer.predict(adv_ds)
adv_y_pred = np.argmax(adv_preds.predictions, axis=-1)
adv_y_true = adv_df['mismatch'].values

adv_correct = (adv_y_pred == adv_y_true).sum()
print(f'Adversarial Test: {adv_correct}/10 correct')
print('Predictions:', adv_y_pred)
print('Ground truth:', adv_y_true)
if adv_correct >= 7:
    print('✓ Adversarial bonus unlocked (≥7/10)!')

## 8. Save Full Predictions CSV

In [ ]:
test_df_out = test_df.copy()
test_df_out['predicted_mismatch'] = y_pred
test_df_out['confidence'] = probs_all[:, 1]
test_df_out[['ticket_id','priority','inferred_severity','predicted_mismatch',
             'mismatch_type','confidence']].to_csv('outputs/predictions.csv', index=False)
print('Predictions saved to outputs/predictions.csv')